# Geodesics and Orbital Dynamics around Black Holes

Using EinsteinPy to simulate particle trajectories and geodesics around black holes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from einsteinpy.metric import Schwarzschild, Kerr
from einsteinpy.geodesic import Geodesic
from einsteinpy.plotting import StaticGeodesicPlotter
from astropy import units as u

print("EinsteinPy Geodesics - Loaded")

## 1. Schwarzschild Geodesics

First, let's simulate geodesics around a Schwarzschild (non-rotating) black hole.

In [ ]:
# Define Schwarzschild metric
M = 1e6 * u.kg  # Mass of black hole
pos = [10.0, np.pi/2, 0.0]  # Initial position (r, theta, phi)
mom = [0.0, 0.0, 1.5]  # Initial momentum

# Create Schwarzschild metric
sch = Schwarzschild(coords="Boyer-Lindquist", M=M)

# Create geodesic
geod = Geodesic(
    metric=sch,
    position=pos,
    momentum=mom,
    steps=1000,
    delta=0.5,
    return_cartesian=True,
    lamb=2000.0,
)

# Integrate
geod.calculate_trajectory()

# Plot
sgpl = StaticGeodesicPlotter()
sgpl.plot(geod)
sgpl.show()

## 2. Circular Orbits around Schwarzschild Black Hole

Calculate the radius of the innermost stable circular orbit (ISCO):

In [ ]:
def isco_radius_schwarzschild(mass):
    """
    Calculate ISCO radius for Schwarzschild black hole.
    
    For Schwarzschild: R_isco = 6 * Rs = 6 * (2GM/c^2) = 12GM/c^2
    """
    from astropy import constants as const
    rs = 2 * const.G * mass / const.c**2
    r_isco = 6 * rs
    return r_isco.to(u.km)

# ISCO for different masses
test_masses = [10 * u.Msun, 36 * u.Msun, 4.1e6 * u.Msun]

print("ISCO Radii:")
print("-" * 50)
for mass in test_masses:
    r_isco = isco_radius_schwarzschild(mass)
    rs = 2 * const.G * mass / const.c**2
    print(f"Mass: {mass:.2e} -> Rs: {rs.to(u.km):.2f}, ISCO: {r_isco:.2f}")

## 3. Kerr Black Hole Geodesics

Now let's simulate geodesics around a rotating (Kerr) black hole.

In [ ]:
# Define Kerr metric with spin parameter a
a = 0.9  # Spin parameter (dimensionless, 0 = Schwarzschild, 1 = extremal)
M = 1e6 * u.kg

# Create Kerr metric
kerr = Kerr(coords="Boyer-Lindquist", M=M, a=a)

# Initial conditions for a prograde orbit
pos = [10.0, np.pi/2, 0.0]
mom = [0.0, 0.0, 1.8]  # Higher angular momentum for Kerr

# Create geodesic
geod_kerr = Geodesic(
    metric=kerr,
    position=pos,
    momentum=mom,
    steps=1000,
    delta=0.5,
    return_cartesian=True,
    lamb=2000.0,
)

# Integrate
geod_kerr.calculate_trajectory()

# Plot
sgpl_kerr = StaticGeodesicPlotter()
sgpl_kerr.plot(geod_kerr)
sgpl_kerr.show()

## 4. Comparison: Schwarzschild vs Kerr ISCO

The ISCO depends on the spin parameter a:

In [ ]:
def isco_radius_kerr(mass, a):
    """
    Calculate ISCO radius for Kerr black hole.
    
    Formula depends on whether orbit is prograde or retrograde.
    
    Prograde: R_isco = Rs * [3 + Z2 - sqrt((3-Z1)(3+Z1+2Z2))]
    where Z1, Z2 are functions of a
    """
    from astropy import constants as const
    rs = 2 * const.G * mass / const.c**2
    
    # Simplified approximation for prograde
    # R_isco ≈ Rs * (3 + Z2 - sqrt((3-Z1)(3+Z1+2Z2)))
    # For simplicity, use the fact that:
    # Schwarzschild (a=0): R_isco = 6 Rs
    # Extremal (a=1): R_isco = 1 Rs (prograde)
    
    # Linear approximation for visualization
    r_isco = rs * (6 - 5*a)  # Approximation
    return r_isco.to(u.km)

# Compare ISCO for different spins
mass = 36 * u.Msun  # GW150914 mass
spins = [0.0, 0.5, 0.9, 0.99]

print(f"ISCO for {mass} Black Hole:")
print("-" * 60)
for a in spins:
    r_isco = isco_radius_kerr(mass, a)
    rs = 2 * const.G * mass / const.c**2
    print(f"Spin a={a:.2f} -> Rs: {rs.to(u.km):.2f}, ISCO: {r_isco:.2f}")

## 5. Orbital Period

Calculate the orbital period at a given radius:

In [ ]:
def orbital_period(mass, radius):
    """
    Calculate orbital period at a given radius (Newtonian approximation).
    
    T = 2π * sqrt(r^3 / GM)
    """
    from astropy import constants as const
    T = 2 * np.pi * np.sqrt(radius**3 / (const.G * mass))
    return T.to(u.s)

# Orbital periods at different radii for Sgr A*
sgrA_mass = 4.1e6 * u.Msun
sgrA_rs = 2 * const.G * sgrA_mass / const.c**2

radii = [5*sgrA_rs, 10*sgrA_rs, 50*sgrA_rs, 100*sgrA_rs]

print(f"Orbital Periods near Sgr A*:")
print("-" * 70)
for r in radii:
    T = orbital_period(sgrA_mass, r)
    print(f"Radius: {r.to(u.km):.2e} -> Period: {T.to(u.hour):.2f}")

## Summary

This notebook covered:
1. Schwarzschild geodesics
2. ISCO calculation
3. Kerr black hole geodesics
4. Spin effects on ISCO
5. Orbital periods

Next: Gravitational wave modeling.